In [1]:
%run ../setup_data.ipynb

No active 'spark' session found to stop.
Starting new Spark session: 'working_with_different_data_type'...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/11 15:21:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


flight path verified (6 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/flight-data/csv
retail path verified (7 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/retail-data
Loading flight data...
Loading retail data...
+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+



In [2]:
spark.sql("show tables in sujeet_db").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+



In [3]:
department_df = spark.sql("select * from sujeet_db.department")
payement_df = spark.sql("""select * from sujeet_db.payment""")

### Grouping

First we specify the column(s) on which we would like to
group, and then we specify the aggregation(s). The first step returns a
**RelationalGroupedDataset**, and the second step returns a DataFrame.

In [4]:
payement_df.show()

+--------------+------------+--------+
|payment_amount|payment_date|store_id|
+--------------+------------+--------+
|       1200.99|  2018-01-18|       1|
|        189.23|  2018-02-15|       1|
|       3332.23|  2021-02-23|       3|
|       9499.49|  2021-03-10|       3|
|        994.11|  2020-03-14|       1|
|        394.93|  2021-01-22|       2|
|         33.43|  2018-03-03|       3|
|       7382.10|  2019-01-11|       2|
|       2929.14|  2020-01-03|       2|
|        499.02|  2020-02-19|       3|
|       3002.43|  2018-02-25|       2|
|        100.99|  2019-03-07|       1|
|        211.65|  2020-02-02|       1|
|        500.73|  2021-01-06|       3|
|        382.92|  2019-02-18|       1|
|        322.34|  2019-03-29|       2|
+--------------+------------+--------+



In [5]:

from pyspark.sql.functions import to_date

formated_payment_df = (

    payement_df
    .withColumn("payement_date", to_date("payment_date"))
)

In [6]:
formated_payment_df.groupBy("store_id").count().show()

+--------+-----+
|store_id|count|
+--------+-----+
|       1|    6|
|       3|    5|
|       2|    5|
+--------+-----+



In [7]:

from pyspark.sql.functions import to_date, sum,count

formated_payment_df.groupBy("store_id").agg(
    sum("payment_amount"),
    count("store_id")

).show()

+--------+-------------------+---------------+
|store_id|sum(payment_amount)|count(store_id)|
+--------+-------------------+---------------+
|       1|            3079.89|              6|
|       3|           13864.90|              5|
|       2|           14030.94|              5|
+--------+-------------------+---------------+



## Window Functions

A group-by takes data, and every row can go only into one grouping.

A window function calculates a return value for every input row of a table based on a group of rows, called a frame.
Each row can fall into one or more frames.

### `groupBy()` vs. `Window` in PySpark

| Feature | `groupBy()` | Window (`over()`) |
| :--- | :--- | :--- |
| **Row Cardinality** | **Many-to-One / Reduction**: Reduces rows to one single summary row per unique group key. | **Row-Preserving**: Preserves the original row count; computes aggregations across a target partition/frame and appends the result to each record. |
| **DataFrame Shape** | Alters the schema and drastically reduces rows (dimensions change). | Keeps the original schema and record count, simply appending new calculated columns (e.g., via `.withColumn()`). |
| **Ordering & Sub-frames** | Does not maintain individual row ordering within groups or relative row boundaries. | Supports fine-grained ordering (`orderBy()`) and relative row boundaries (`rowsBetween()` / `rangeBetween()`) for running totals, moving averages, and ranks. |
| **Use Cases** | Summary reports, group-level totals, distinct categorical counts. | Cumulative metrics (running totals), moving averages, row-level rankings (`dense_rank`, `row_number`), and comparing individual rows against group baselines. |

In [8]:
department_df.show()

+---------+----------+------+
|     name|department|salary|
+---------+----------+------+
|     Jeff| Marketing|  3000|
|    Kumar| Marketing|  2000|
|  Michael|     Sales|  4600|
|   Robert|     Sales|  4100|
|    Maria|   Finance|  3000|
|      Jen|   Finance|  3900|
|    James|     Sales|  3000|
|    Scott|   Finance|  3300|
|    James|     Sales|  3000|
|     Saif|     Sales|  4100|
|   Daniel|      NULL|  NULL|
|Christian|      NULL|  NULL|
+---------+----------+------+



In [9]:
from pyspark.sql.functions import *

(
    department_df
    .groupBy(col("department"))
    .agg(max(col("salary")))
    .show(10,False)
)

+----------+-----------+
|department|max(salary)|
+----------+-----------+
|Marketing |3000       |
|Sales     |4600       |
|Finance   |3900       |
|NULL      |NULL       |
+----------+-----------+



In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc_nulls_last, max, avg, round

win_spec = Window.partitionBy("department").orderBy(desc_nulls_last(col("salary")))

(
    department_df
    .withColumn("max_salary", max(col("salary")).over(win_spec) )
     .withColumn("avg_salary", round(avg(col("salary")).over(win_spec),2) )
).show()

+---------+----------+------+----------+----------+
|     name|department|salary|max_salary|avg_salary|
+---------+----------+------+----------+----------+
|   Daniel|      NULL|  NULL|      NULL|      NULL|
|Christian|      NULL|  NULL|      NULL|      NULL|
|      Jen|   Finance|  3900|      3900|    3900.0|
|    Scott|   Finance|  3300|      3900|    3600.0|
|    Maria|   Finance|  3000|      3900|    3400.0|
|     Jeff| Marketing|  3000|      3000|    3000.0|
|    Kumar| Marketing|  2000|      3000|    2500.0|
|  Michael|     Sales|  4600|      4600|    4600.0|
|   Robert|     Sales|  4100|      4600|   4266.67|
|     Saif|     Sales|  4100|      4600|   4266.67|
|    James|     Sales|  3000|      4600|    3760.0|
|    James|     Sales|  3000|      4600|    3760.0|
+---------+----------+------+----------+----------+



-  row_number(): Assigns a unique, sequential integer (1, 2, 3, 4...) to every row regardless of ties. Tied rows get arbitrary sequence numbers unless you add a tie-breaker column to orderBy().

-  rank(): Assigns the same rank to tied values, then leaves a gap in the sequence based on how many rows were tied (1, 2, 2, 4...).

- dense_rank(): Assigns the same rank to tied values, with no gaps in the sequence (1, 2, 2, 3...).


In [11]:
## rank, dense_rank, row_number
from pyspark.sql.functions import rank, dense_rank, row_number

(
    department_df
    .withColumn("rank", rank().over(win_spec) )
     .withColumn("dense_rank", dense_rank().over(win_spec))
     .withColumn("row_number", row_number().over(win_spec))
).show()


+---------+----------+------+----+----------+----------+
|     name|department|salary|rank|dense_rank|row_number|
+---------+----------+------+----+----------+----------+
|   Daniel|      NULL|  NULL|   1|         1|         1|
|Christian|      NULL|  NULL|   1|         1|         2|
|      Jen|   Finance|  3900|   1|         1|         1|
|    Scott|   Finance|  3300|   2|         2|         2|
|    Maria|   Finance|  3000|   3|         3|         3|
|     Jeff| Marketing|  3000|   1|         1|         1|
|    Kumar| Marketing|  2000|   2|         2|         2|
|  Michael|     Sales|  4600|   1|         1|         1|
|   Robert|     Sales|  4100|   2|         2|         2|
|     Saif|     Sales|  4100|   2|         2|         3|
|    James|     Sales|  3000|   4|         3|         4|
|    James|     Sales|  3000|   4|         3|         5|
+---------+----------+------+----+----------+----------+



In [12]:
## let say we lowest max of each department (by default asc)

win_spec = Window.partitionBy("department") .orderBy(col("salary"))

(
department_df
.withColumn("max_salary", max("salary").over(win_spec))
.withColumn("total_salary", sum("salary").over(win_spec)) ## not working b/c default to rangeBetween(UnboundPrecedding, currentRow)
.withColumn("rank", rank().over(win_spec))
# .filter(col("rank") == 1)
).show()


"""
1. rank require orderBy # it need to know which row comes first
2. if we don't use orderBy, then default would rowBetween(UnboundPrecedding, UnboundFallowing) and can see the sum of as total for each row

"""

+---------+----------+------+----------+------------+----+
|     name|department|salary|max_salary|total_salary|rank|
+---------+----------+------+----------+------------+----+
|   Daniel|      NULL|  NULL|      NULL|        NULL|   1|
|Christian|      NULL|  NULL|      NULL|        NULL|   1|
|    Maria|   Finance|  3000|      3000|        3000|   1|
|    Scott|   Finance|  3300|      3300|        6300|   2|
|      Jen|   Finance|  3900|      3900|       10200|   3|
|    Kumar| Marketing|  2000|      2000|        2000|   1|
|     Jeff| Marketing|  3000|      3000|        5000|   2|
|    James|     Sales|  3000|      3000|        6000|   1|
|    James|     Sales|  3000|      3000|        6000|   1|
|   Robert|     Sales|  4100|      4100|       14200|   3|
|     Saif|     Sales|  4100|      4100|       14200|   3|
|  Michael|     Sales|  4600|      4600|       18800|   5|
+---------+----------+------+----------+------------+----+



"\n1. rank require orderBy # it need to know which row comes first\n2. if we don't use orderBy, then default would rowBetween(UnboundPrecedding, UnboundFallowing) and can see the sum of as total for each row\n\n"

In [13]:
from pyspark.sql import Window
from pyspark.sql.functions import col, max, sum, rank

# 1. Partition-only window (defaults to entire partition: unboundedPreceding to unboundedFollowing)
dept_win = Window.partitionBy("department")

# 2. Ordered window (required for ranking; note .desc() if you want rank 1 = highest salary)
rank_win = Window.partitionBy("department").orderBy(col("salary").desc())

(
    department_df
    .withColumn("max_salary", max("salary").over(dept_win))
    .withColumn("total_salary", sum("salary").over(dept_win))
    .withColumn("rank", rank().over(rank_win))
    .filter(col("rank") == 1)
).show()


+---------+----------+------+----------+------------+----+
|     name|department|salary|max_salary|total_salary|rank|
+---------+----------+------+----------+------------+----+
|   Daniel|      NULL|  NULL|      NULL|        NULL|   1|
|Christian|      NULL|  NULL|      NULL|        NULL|   1|
|      Jen|   Finance|  3900|      3900|       10200|   1|
|     Jeff| Marketing|  3000|      3000|        5000|   1|
|  Michael|     Sales|  4600|      4600|       18800|   1|
+---------+----------+------+----------+------------+----+



 # PySpark Window Frames: `rowsBetween`, `rangeBetween`, and Boundaries

A **Window function** calculates a return value for every input row of a table based on a group of related rows called a **frame**. Unlike `groupBy()`, where every row can go into only one grouping, a window function preserves all original rows, and each row can fall into one or more frames.

### 1. Three Core Components of a `WindowSpec`
1. **`partitionBy()`**: Defines how rows are broken up into independent groups/partitions (e.g., `department`).
2. **`orderBy()`**: Defines how rows are ordered within each partition (e.g., `salary`).
3. **Frame Specification (`rowsBetween` or `rangeBetween`)**: States which rows are included in the frame relative to the current input row.

---

### 2. Boundary Pointers
| Boundary Constant / Value | SQL Equivalent | Meaning |
| :--- | :--- | :--- |
| **`Window.unboundedPreceding`** | `UNBOUNDED PRECEDING` | Starts at the very **first row** of the partition. |
| **`Window.currentRow`** (`0`) | `CURRENT ROW` | The current row (or value) being evaluated. |
| **`Window.unboundedFollowing`** | `UNBOUNDED FOLLOWING` | Ends at the very **last row** of the partition. |
| **Negative Integer (`-K`)** | `K PRECEDING` | $K$ rows (or $K$ value units) **before** the current row. |
| **Positive Integer (`+J`)** | `J FOLLOWING` | $J$ rows (or $J$ value units) **after** the current row. |

---

### 3. `rowsBetween` vs. `rangeBetween`
* **`rowsBetween(start, end)` (Physical Row Offset)**: Counts physical row positions step-by-step based on the sort order. Even if two rows have tied values (`3000` and `3000`), they are processed one at a time.
* **`rangeBetween(start, end)` (Logical Value Offset)**: Evaluates the **actual numeric value** of the `orderBy` column.
  * All tied rows (peers with identical values like `3000` or `4100`) are treated as a single group at `Window.currentRow`.
  * Numeric offsets like `.rangeBetween(-500, 500)` include all rows in the partition whose column value falls within `[current_value - 500, current_value + 500]`.

> **Default Spark Frame Rules:**
> * **Without `orderBy`**: Defaults to `rowsBetween(unboundedPreceding, unboundedFollowing)` (entire partition).
> * **With `orderBy` (and no frame specified)**: Defaults to `rangeBetween(unboundedPreceding, currentRow)`.

In [14]:
from pyspark.sql.window import Window


# Base window ordered by salary ascending
base_win = Window.partitionBy("department").orderBy(col("salary").asc_nulls_last())

# 1. rowsBetween: Physical row-by-row running total
rows_unbounded_to_curr = base_win.rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 2. rangeBetween: Value-based running total (combines duplicates immediately!)
range_unbounded_to_curr = base_win.rangeBetween(Window.unboundedPreceding, Window.currentRow)

# 3. Full Partition: Unbounded Preceding to Unbounded Following (Total Dept Salary)
full_dept_win = base_win.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

(
    department_df
    .withColumn("rows_running_sum", sum("salary").over(rows_unbounded_to_curr))
    .withColumn("range_running_sum", sum("salary").over(range_unbounded_to_curr))
    .withColumn("dept_total_sum", sum("salary").over(full_dept_win))
    .orderBy("department", "salary")
    .show(truncate=False)
)

+---------+----------+------+----------------+-----------------+--------------+
|name     |department|salary|rows_running_sum|range_running_sum|dept_total_sum|
+---------+----------+------+----------------+-----------------+--------------+
|Daniel   |NULL      |NULL  |NULL            |NULL             |NULL          |
|Christian|NULL      |NULL  |NULL            |NULL             |NULL          |
|Maria    |Finance   |3000  |3000            |3000             |10200         |
|Scott    |Finance   |3300  |6300            |6300             |10200         |
|Jen      |Finance   |3900  |10200           |10200            |10200         |
|Kumar    |Marketing |2000  |2000            |2000             |5000          |
|Jeff     |Marketing |3000  |5000            |5000             |5000          |
|James    |Sales     |3000  |3000            |6000             |18800         |
|James    |Sales     |3000  |6000            |6000             |18800         |
|Robert   |Sales     |4100  |10100      

## Sliding Windows in PySpark (`rowsBetween` vs. `rangeBetween`)

Unlike a running total (which starts at `unboundedPreceding` and grows larger with every row)[cite: 3], a **sliding window** defines a moving boundary around the current row[cite: 3, 4]. Each input row can fall into multiple overlapping frames as the window slides down the partition[cite: 3].

### 1. Two Ways to Slide a Window

| Method | Syntax Example | How the Window Slides |
| :--- | :--- | :--- |
| **Physical Row Slide (`rowsBetween`)** | `.rowsBetween(-1, 1)` | Slides by **physical row position**: includes 1 row before (`-1`), the current row (`0`), and 1 row after (`+1`). At the very top or bottom of a partition, if a neighbor doesn't exist, the frame simply includes fewer rows. |
| **Logical Value Slide (`rangeBetween`)** | `.rangeBetween(-500, 500)` | Slides by **column value**: if the current row's `salary` is $S$, the frame includes all rows in the partition whose salary falls within $[S - 500, S + 500]$. |

### 2. Common Sliding Frame Patterns (`rowsBetween`)
* **`.rowsBetween(-1, 0)` (1 Before + Current)**: Compares or averages the current row with the immediately preceding row.
* **`.rowsBetween(-1, 1)` (Centered 3-Row Moving Window)**: Includes the previous row, current row, and next row.
* **`.rowsBetween(0, 1)` (Current + 1 After)**: Looks forward from the current row to the next row.

In [17]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, avg, round, collect_list

# Base window: partitioned by department, ordered by salary ascending
base_win = Window.partitionBy("department").orderBy(col("salary").asc())

# 1. Physical Sliding Window: 1 row before (-1), current row (0), 1 row after (+1)
row_slide_win = base_win.rowsBetween(-1, 1)

# 2. Value Sliding Window: Salaries within [current_salary - 500, current_salary + 500]
range_slide_win = base_win.rangeBetween(-500, 500)

(
    department_df
    .where(col("department").isNotNull())
    .withColumn("rows_[-1,1]_values", collect_list("salary").over(row_slide_win))
    .withColumn("rows_[-1,1]_sum", sum("salary").over(row_slide_win))
    .withColumn("range_[-500,+500]_values", collect_list("salary").over(range_slide_win))
    .withColumn("range_[-500,+500]_sum", sum("salary").over(range_slide_win))
    .orderBy("department", "salary")
    .show(truncate=False)
)

+-------+----------+------+------------------+---------------+------------------------+---------------------+
|name   |department|salary|rows_[-1,1]_values|rows_[-1,1]_sum|range_[-500,+500]_values|range_[-500,+500]_sum|
+-------+----------+------+------------------+---------------+------------------------+---------------------+
|Maria  |Finance   |3000  |[3000, 3300]      |6300           |[3000, 3300]            |6300                 |
|Scott  |Finance   |3300  |[3000, 3300, 3900]|10200          |[3000, 3300]            |6300                 |
|Jen    |Finance   |3900  |[3300, 3900]      |7200           |[3900]                  |3900                 |
|Kumar  |Marketing |2000  |[2000, 3000]      |5000           |[2000]                  |2000                 |
|Jeff   |Marketing |3000  |[2000, 3000]      |5000           |[3000]                  |3000                 |
|James  |Sales     |3000  |[3000, 3000]      |6000           |[3000, 3000]            |6000                 |
|James  |S

## Understanding `lag()` and `lead()` in PySpark

When working with a `WindowSpec` partitioned by a column (e.g., `department`) and ordered by a column (e.g., `salary`), `lag()` and `lead()` let you fetch values from neighboring physical rows relative to the current row:

| Function | Direction | Syntax | Parameters |
| :--- | :--- | :--- | :--- |
| **`lag(col, offset, default)`** | **Looks Backward ($\uparrow$)** | `lag("salary", 1, 0)` | • `col`: Column to fetch value from.<br>• `offset` *(default `1`)*: How many physical rows **before** the current row to look.<br>• `default` *(default `None`)*: Value returned if there is no previous row (e.g., on the 1st row of a partition). |
| **`lead(col, offset, default)`** | **Looks Forward ($\downarrow$)** | `lead("salary", 1, 0)` | • `col`: Column to fetch value from.<br>• `offset` *(default `1`)*: How many physical rows **after** the current row to look.<br>• `default` *(default `None`)*: Value returned if there is no next row (e.g., on the last row of a partition). |

> **Important Note:**
> * **`orderBy()` is mandatory**: You must specify `.orderBy()` inside your `Window` specification so Spark knows what "before" and "after" mean.
> * **No `rowsBetween` / `rangeBetween` needed**: `lag()` and `lead()` use their own `offset` argument rather than a window frame specification.

In [18]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, lead

# Define the Window: Partition by department, Order by salary ascending
win_spec = Window.partitionBy("department").orderBy(col("salary").asc())

(
    department_df
    .where(col("department").isNotNull())
    # 1. Look 1 row BACKWARD (default = None if first row in department)
    .withColumn("prev_salary_lag1", lag("salary", 1).over(win_spec))
    
    # 2. Look 1 row FORWARD (default = None if last row in department)
    .withColumn("next_salary_lead1", lead("salary", 1).over(win_spec))
    
    # 3. Look 2 rows BACKWARD with a default fallback value of 0 instead of NULL
    .withColumn("lag_2_with_default", lag("salary", 2, 0).over(win_spec))
    
    # 4. Practical Use Case: Difference between current salary and previous salary
    .withColumn("diff_from_prev", col("salary") - col("prev_salary_lag1"))
    .orderBy("department", "salary")
    .show(truncate=False)
)

+-------+----------+------+----------------+-----------------+------------------+--------------+
|name   |department|salary|prev_salary_lag1|next_salary_lead1|lag_2_with_default|diff_from_prev|
+-------+----------+------+----------------+-----------------+------------------+--------------+
|Maria  |Finance   |3000  |NULL            |3300             |0                 |NULL          |
|Scott  |Finance   |3300  |3000            |3900             |0                 |300           |
|Jen    |Finance   |3900  |3300            |NULL             |3000              |600           |
|Kumar  |Marketing |2000  |NULL            |3000             |0                 |NULL          |
|Jeff   |Marketing |3000  |2000            |NULL             |0                 |1000          |
|James  |Sales     |3000  |NULL            |3000             |0                 |NULL          |
|James  |Sales     |3000  |3000            |4100             |0                 |0             |
|Robert |Sales     |4100  |300

In [19]:
spark.stop()